# Trinity TEFB MC — Kaggle Benchmark Task

**Brain Zone:** Executive / dlPFC + ACC  
**Questions:** 1,805 MC (4-choice) + 595 factual  
**Dataset:** [playra/trinity-cognitive-probes-tefb-mc](https://www.kaggle.com/datasets/playra/trinity-cognitive-probes-tefb-mc)  
**Framework:** `kaggle-benchmarks` SDK


In [ ]:
# [TEFB] Step 0: Install dependencies
!pip install protobuf==5.29.6 --quiet
!pip install -q kaggle-benchmarks kaggle
import kaggle_benchmarks as kbench
print(f"\U0001f539 [TEFB] Step 0: kbench version: {kbench.__version__}")

In [ ]:
# [TEFB] Step 1: Imports and configuration
import os
os.environ["RENDER_SUBRUNS"] = "False"

import kaggle_benchmarks as kbench
import kaggle
import pandas as pd
from dataclasses import dataclass
import glob
import re
print("\U0001f539 [TEFB] Step 1: Imports successful")

In [ ]:
# [TEFB] Step 2: Download dataset and discover CSV
print("\U0001f539 [TEFB] Step 2: Downloading dataset")
print(f"Dataset URL: https://www.kaggle.com/datasets/playra/trinity-cognitive-probes-tefb-mc")
os.makedirs('/kaggle/working/datasets', exist_ok=True)

kaggle.api.dataset_download_files(
    'playra/trinity-cognitive-probes-tefb-mc',
    path='/kaggle/working/datasets',
    unzip=True
)

csv_files = glob.glob('/kaggle/working/datasets/**/*.csv', recursive=True)
print(f"\U0001f539 [TEFB] Step 2: Available CSVs: {csv_files}")

In [ ]:
# [TEFB] Step 3: Load CSV, validate MC answers, prepare eval set
csv_path = csv_files[0]
print(f"\U0001f539 [TEFB] Step 3: Using: {csv_path}")

df = pd.read_csv(csv_path)
print(f"\U0001f539 [TEFB] Loaded rows: {len(df)}")
print(f"\U0001f539 [TEFB] Columns: {list(df.columns)}")

# Filter MC rows only
if 'question_type' in df.columns:
    df = df[df['question_type'] == 'mc'].copy()
print(f"\U0001f539 [TEFB] MC rows: {len(df)}")

# Sanity check: validate answer column contains only A-D
valid_answers = df['answer'].astype(str).str.strip().str.upper().str.match(r'^[A-D]$')
mc_ratio = valid_answers.mean()
print(f"\U0001f539 [TEFB] MC answer validity: {mc_ratio:.1%}")
assert mc_ratio > 0.95, f'MC answer ratio too low: {mc_ratio:.1%} — check CSV data quality'

# Prepare eval DataFrame
eval_df = df.rename(columns={'answer': 'expected_answer'})
eval_df = eval_df[['question', 'choices', 'expected_answer']].reset_index(drop=True)
print(f"\U0001f539 [TEFB] Eval size: {len(eval_df)}")

In [ ]:
# [TEFB] Step 4: Structured output schema for MC answers
@dataclass
class MCAnswer:
    answer: str

print("\U0001f539 [TEFB] Step 4: Schema defined")

In [ ]:
# [TEFB] Step 5: Inner task — evaluate a single MC question
@kbench.task(name='tefb_single_mc', store_task=False)
def tefb_single_mc(llm, question: str, choices: str, expected_answer: str) -> bool:
    prompt = f"""{question}

{choices}

Answer with ONLY ONE letter (A, B, C, or D).
Do not explain. Return exactly one character."""
    resp = llm.prompt(prompt, schema=MCAnswer)
    got = resp.answer.strip().upper()[:1]
    exp = str(expected_answer).strip().upper()[:1]
    return got == exp

print("\U0001f539 [TEFB] Step 5: Inner task defined")

In [ ]:
# [TEFB] Step 6: Outer benchmark — aggregate accuracy over all MC questions
@kbench.task(name='tefb_mc_benchmark', description='Executive Function Battery')
def tefb_mc_benchmark(llm) -> float:
    with kbench.client.enable_cache():
        runs = tefb_single_mc.evaluate(
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=2,
            timeout=180,
            max_attempts=1,
            remove_run_files=True,
        )
    df_res = runs.as_dataframe()
    valid = df_res[df_res['result'].notna()]
    correct = int(valid['result'].sum())
    total = len(eval_df)
    acc = float(valid['result'].mean())
    print(f"\U0001f539 [TEFB] Valid: {len(valid)}/{total}, Correct: {correct}")
    kbench.assertions.assert_true(
        True,
        expectation=f'TEFB MC accuracy: {acc:.2%} ({correct}/{total})'
    )
    return acc

print("\U0001f539 [TEFB] Step 6: Benchmark task defined")

# [TEFB] Step 7: Execute and submit
run = tefb_mc_benchmark.run()
print(f"\n\U0001f3c6 [TEFB] MC Accuracy: {run.result:.1%}")
%choose tefb_mc_benchmark